In [6]:
import sys
import os
import numpy as np
import skfuzzy as fuzzy
from skfuzzy import control as ctrl

In [7]:
def criar_sistema_triagem_uti():
    """Cria o sistema fuzzy de triagem de UTI e retorna o ControlSystemSimulation."""
    fc = ctrl.Antecedent(np.arange(40, 161, 1), "freq_cardiaca")
    spo2 = ctrl.Antecedent(np.arange(70, 101, 1), "spo2")
    risco = ctrl.Consequent(np.arange(0, 101, 1), "risco", defuzzify_method="centroid")

    fc["bradicardia"] = fuzzy.trimf(fc.universe, [40, 40, 60])
    fc["normal"] = fuzzy.trimf(fc.universe, [50, 75, 100])
    fc["taquicardia"] = fuzzy.trapmf(fc.universe, [90, 110, 160, 160])

    spo2["critica"] = fuzzy.trapmf(spo2.universe, [70, 70, 88, 91])
    spo2["limitrofe"] = fuzzy.trimf(spo2.universe, [89, 93, 96])
    spo2["normal"] = fuzzy.trapmf(spo2.universe, [94, 97, 100, 100])

    risco["baixo"] = fuzzy.trimf(risco.universe, [0, 0, 35])
    risco["moderado"] = fuzzy.trimf(risco.universe, [25, 50, 75])
    risco["critico"] = fuzzy.trapmf(risco.universe, [65, 85, 100, 100])

    regras = [
        ctrl.Rule(spo2["critica"], risco["critico"]),
        ctrl.Rule(fc["bradicardia"] & spo2["limitrofe"], risco["critico"]),
        ctrl.Rule(fc["taquicardia"] & spo2["limitrofe"], risco["critico"]),
        ctrl.Rule(fc["normal"] & spo2["limitrofe"], risco["moderado"]),
        ctrl.Rule(fc["normal"] & spo2["normal"], risco["baixo"]),
    ]
    return ctrl.ControlSystemSimulation(ctrl.ControlSystem(regras))


def avaliar_paciente(simulador, bpm, spo2_val):
    """Valida as leituras, executa a inferência e retorna o risco (0-100)."""
    if not (40 <= bpm <= 160):
        raise ValueError(f"bpm fora do intervalo fisiológico [40, 160]: {bpm}")
    if not (70 <= spo2_val <= 100):
        raise ValueError(f"SpO2 fora do intervalo [70, 100]: {spo2_val}")
    simulador.input["freq_cardiaca"] = bpm
    simulador.input["spo2"] = spo2_val
    simulador.compute()
    return float(simulador.output["risco"])


if __name__ == "__main__":
    sim = criar_sistema_triagem_uti()
    print("Sistema de UTI instanciado com sucesso!")
    for b, s in [(75, 98), (75, 95), (75, 89)]:
        print(f"FC={b} SpO2={s} -> risco={avaliar_paciente(sim, b, s):.2f}")


Sistema de UTI instanciado com sucesso!
FC=75 SpO2=98 -> risco=11.67
FC=75 SpO2=95 -> risco=35.71
FC=75 SpO2=89 -> risco=85.57
